In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]  # repository root; run from research/notebooks with pidu installed

Steps to do:
1. Run generate-ypred.py with debug configuration "Debugger: generate-ypred" (this will generate the ypred for the dook case and save it in the outputs folder). This will export the file to:
f"casestudies/case{args.usecase}/outputs/{args.name_run}/{args.set}", f"u{args.usecase}_{args.set}_{args.name_run}"

2. Provided the simulated data, use the data from "experimental_data/STO/system1_refl".

3. 

In [ ]:
import numpy as np
import os
from pathlib import Path

dir_path = (ROOT / 'data/experimental_data/STO/system1_refl')
dir_path2 = (ROOT / 'data/experimental_data/STO/system2_0_retrans')
dir_path3 = (ROOT / 'data/experimental_data/STO/system3_0_imtrans')

data = {}
data2 = {}
data3 = {}

In [ ]:
# Load all measurements from the directory (refl)

measurements_meas = []
meas_x = []

for file in os.listdir(dir_path):
    meas = np.loadtxt(dir_path / file, skiprows=1)
    measurements_meas.append(meas)
    meas_x.append(meas[:, 0])

measurements_meas = np.array(measurements_meas) # First measurement with x-axis
meas_x = np.array(meas_x) # x-axis of all measurements
measurements_meas = np.concatenate((meas_x[0][None,:], measurements_meas[:,:,1]), axis=0)

print(f"Number of measurements: {len(measurements_meas)}")
print(f"Shape of first measurement: {measurements_meas.shape}")
print(f"Shape of x-axis: {meas_x.shape}")



In [ ]:
# Load all simulated from excel file
import pandas as pd

measurements_sim = pd.read_excel(str(ROOT / 'outputs/case2/dook/exp/u2_exp_dook.xlsx')).to_numpy().T

print(f"Shape of simulated measurement: {measurements_sim.shape}")

In [ ]:
# plot a signal
import matplotlib.pyplot as plt
%matplotlib qt

sample_idx = 45

assert sample_idx > 0, "Sample index should be greater than 0, since measurements_meas is 0-indexed and measurements_sim is 1-indexed."

fig, ax = plt.subplots(1,1)

ax.plot(measurements_sim[0], measurements_sim[sample_idx], label='Simulated')
ax.plot(measurements_meas[0], measurements_meas[sample_idx], label='Measured')
ax.vlines(30, 0, 1, colors='r', linestyles='dashed')

ax.set(
  ylim=(0, 1.1),
  xlim=(0, 1024)
)


In [ ]:
fit = measurements_sim    # 
meas = measurements_meas  # 

print((fit.shape, meas.shape))

In [ ]:
# Interpolate
import numpy as np
from scipy.interpolate import interp1d

res = [meas[0]] # x-axis
fit_inter = [meas[0]] # x-axis


for i in range(1, measurements_meas.shape[0]):
    f_interp = interp1d(fit[0], fit[i], kind='linear', fill_value="extrapolate")
    si_fit_interp = f_interp(meas[0])
    residual = meas[i] - si_fit_interp
    res.append(residual)
    fit_inter.append(si_fit_interp)

res = np.array(res)
fit_inter = np.array(fit_inter)

print(f"Residual shape: {res.shape}, Fit interp shape: {fit_inter.shape}")

In [ ]:
# plot diff
import matplotlib.pyplot as plt

boundaries = (None, None)

fig, ax = plt.subplots(3,1)

ax[0].plot(fit[0], fit[1], label='Fitted')            # Fitted via simulation
ax[0].plot(meas[0], meas[1], label='Measured')        # Measured
ax[0].plot(res[0], np.abs(res[1]), label='Residual') # 
ax[1].plot(res[0], res[1], label='Residual')          #
ax[2].plot(res[0], res[1]/fit_inter[1], label='Abs Residual')  #
ax[1].axhline(np.mean(res[1][15:220]), color='r', linestyle='--', label='Mean Residual')

print(np.mean(res[1][15:220]), np.std(res[1][15:220]), meas[0,220])

ax[0].set(
  ylim=(0, 1.1),
  xlim=boundaries,
)
ax[1].set(
  ylim=(-0.4, 0.4),
  xlim=boundaries,
)
ax[2].set(
  ylim=(-0.4, 0.4),
  xlim=boundaries,
)

# data_to_export = np.column_stack((meas[:,0], meas[:,1], si_fit_interp, np.abs(residual), residual/si_fit_interp))
# export_path = dir_path / f"{str(sample) if sample is not None else 'si'}_residuals.dat"
# np.savetxt(export_path, data_to_export, header="Wavenumbers (cm-1) Si(meas) Si(fit) Residual Per_Residual", fmt="%.6e", comments="")

# print(f"Saved data to {export_path}")

# data[str(sample) if sample is not None else 'si'] = data_to_export

In [ ]:
# # Create difference matrix (.xlsx) with [N_samples, N_wavenumbers]
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

data = {}
data['refl_x'] = pd.DataFrame(meas_x)
data['refl_y'] = pd.DataFrame(measurements_meas[1:,:]) # 
data['refl_diff'] = pd.DataFrame(res[1:,:]) #

print(f"Data keys: {data.keys()}, Data shape: {[data[key].shape for key in data.keys()]} ({[type(data[key]) for key in data.keys()]})")

# for key_id, key in enumerate(data.keys()):
#   xlsx_to_export[int(key_id)+1, :] = data[key][:, 4] # Provide relative error
#   xlsx_to_export[int(key_id)+1, :] = data[key][:, 3] # Provide absolute error
#   ax.plot(xlsx_to_export[0,:], np.abs(xlsx_to_export[int(key_id)+1, :]), label=str(key_id))          #
# ax.set(
#   xlim=(0, 120),
#   ylim=(-1, 1),
# )
# ax.legend()

# with open((ROOT / 'data/experimental_data/STO') / "difference_refl.xlsx", "w") as f:
with pd.ExcelWriter((ROOT / 'data/experimental_data/STO') / "difference_refl.xlsx") as f:
  for key, item in data.items():
    item.to_excel(f, sheet_name=key, index=False)

In [ ]:
# Plot average residual
import numpy as np

fig, ax = plt.subplots(1,1)

ax.plot(data['refl_x'].iloc[0], np.mean(data['refl_diff'], axis=0))

ax.set(
  ylim=(-0.2, 0.2),
  xlim=(0, 1000)
)

In [ ]:
import sys

import torch
from pidu.data.utils import  generate_noise_profile_uc3, np2pt

from torch import float32 as pt_float32, device as pt_device

meas = pd.read_excel((ROOT / 'data/experimental_data/metastructure/difference_data2.xlsx'), sheet_name=None, index_col=0)['Sheet1'].to_numpy().T

wn = np2pt(meas[:,0], dtype=pt_float32, device=pt_device("cpu"))
bins = torch.arange(wn[0], wn[-1], 1, dtype=pt_float32, device=pt_device("cpu"))

_dict, wn_mask = generate_noise_profile_uc3(
  (ROOT / 'data/experimental_data/metastructure/difference_data2.xlsx'), 
  wn=wn, 
  bins=bins, 
  n_samples=meas.shape[0], 
  dtype=torch.float32, 
  device='cpu'
)

print(_dict['system1'].shape)

fig, ax = plt.subplots(1,1)

In [ ]:
# measurements

refl = np.loadtxt(str(ROOT / 'data/experimental_data/STO/system1_refl/refl_45.dat'), skiprows=0)
t_real = np.loadtxt(str(ROOT / 'data/experimental_data/STO/system2_0_retrans/retrans_45.dat'), skiprows=0)
t_imag = np.loadtxt(str(ROOT / 'data/experimental_data/STO/system2_1_imtrans/imtrans_45.dat'), skiprows=0)

# -- LOAD PARAMETERS --

In [ ]:
import os, sys
import matplotlib.pyplot as plt
import torch


from pidu.data.physics import TmMethodPyTorch
from pidu.data.simulation import SimulationDatasetBatch
from pidu.data.utils import pt2np

wn_sys1 = torch.tensor(refl[:,0], device='cuda', dtype=torch.float32)

p = torch.tensor([[
  [0, 0.006061516, 5.382954, 0, 0, 72.029396, 1610.081514, 9.833616, 0,0,0, 173.34172, 285.554093, 2.026391, 540.720618, 623.30489, 19.625357],
]], device=wn_sys1.device, dtype=wn_sys1.dtype)

model = TmMethodPyTorch()
output = model(p, wn_sys1)

R = pt2np(output['R'])

# -- system2 --
wn_sys2 = torch.tensor(t_real[:,0], device='cuda', dtype=torch.float32)

p = torch.tensor([[
  [0, 0.00275, 5.382954, 0, 0, 72.029396, 1610.081514, 9.833616, 0,0,0, 173.34172, 285.554093, 2.026391, 540.720618, 623.30489, 19.625357],
]], device=wn_sys2.device, dtype=wn_sys2.dtype)

model = TmMethodPyTorch()
output = model(p, wn_sys2)

t_fd = pt2np(output['t_fd'])

In [ ]:
from tkinter import Y


fig, ax = plt.subplots(3,1)

ax[0].plot(refl[:,0], refl[:,1], label='Refl')
ax[0].plot(refl[:,0], R[0], label='Refl Simulated')

ax[1].plot(t_real[:,0], t_real[:,1], label='Real')
ax[1].plot(t_real[:,0], t_fd[0].real, label='Real')
ax[2].plot(t_imag[:,0], t_imag[:,1], label='Imag')
ax[2].plot(t_imag[:,0], t_fd[0].imag, label='Imag')

ax[1].set(
  xlim=(3.5,80),
  ylim=(-0.5,0.5)
)
ax[2].set(
  xlim=(3.5,80),
  ylim=(-0.5,0.5)
)

In [ ]:
np.savetxt(str(ROOT / 'data/experimental_data/STO/difference_refl.dat'), np.column_stack((refl[:,0], refl[:,1], R[0], refl[:,1] - R[0])), header="Wavenumbers (cm-1) Refl(meas) Refl(sim) Residual", fmt="%.6e", comments="")
np.savetxt(str(ROOT / 'data/experimental_data/STO/difference_ret.dat'), np.column_stack((t_real[:,0], t_real[:,1], t_fd[0].real, t_real[:,1].real - t_fd[0].real)), header="Wavenumbers (cm-1) T(meas) T(sim) Residual", fmt="%.6e", comments="")
np.savetxt(str(ROOT / 'data/experimental_data/STO/difference_imt.dat'), np.column_stack((t_imag[:,0], t_imag[:,1], t_fd[0].imag, t_imag[:,1] - t_fd[0].imag)), header="Wavenumbers (cm-1) ImT(meas) ImT(sim) Residual", fmt="%.6e", comments="")

In [ ]:
path_data = str(ROOT / 'data/experimental_data/STO/difference_data.xlsx')
path_data2 = str(ROOT / 'data/experimental_data/STO/difference_refl.xlsx')

df = pd.read_excel(path_data, sheet_name=None)
df2 = pd.read_excel(path_data2, sheet_name=None)

In [ ]:
print(df['real_diff'].to_numpy().shape)
print(df['real_diff'].to_numpy()[:,1])

In [ ]:
print(df.keys())
print(df2.keys())

noise_profile = {}
noise_profile['r'] = np.column_stack((df2['refl_x'].to_numpy()[0], df2['refl_diff'].to_numpy().mean(axis=0), df2['refl_diff'].to_numpy().std(axis=0)))
noise_profile['t_real'] = np.column_stack((df['real_x'].to_numpy()[:,1], df['real_diff'].to_numpy()[:,1:].mean(axis=1), df['real_diff'].to_numpy()[:,1:].std(axis=1)))
noise_profile['t_imag'] = np.column_stack((df['imag_x'].to_numpy()[:,1], df['imag_diff'].to_numpy()[:,1:].mean(axis=1), df['imag_diff'].to_numpy()[:,1:].std(axis=1)))

with pd.ExcelWriter((ROOT / 'data/experimental_data/STO') / "noise_profiles.xlsx") as f:
  for key, item in noise_profile.items():
    pd.DataFrame(item, columns=['Wavenumber', 'Mean Residual', 'Std Residual']).to_excel(f, sheet_name=key, index=False)
    print(item.shape)